In [ ]:
# === Q5: Clean -> Engineer -> Means -> OLS -> T-tests ===
import re
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import ttest_1samp

# -------- 0) Make a safe copy & clean column names --------
df = pd.read_excel('Car_DataSet.xlsx')

df = df.copy()
df.columns = [re.sub(r'[^A-Za-z0-9]+', '', c).strip('').lower() for c in df.columns]
# Example mappings after cleaning:
# "Mileage Driven"      -> "mileage_driven"
# "Price in £"          -> "price_in"
# "Registration Year"   -> "registration_year"
# "Engine ( in Litres)" -> "engine_in_litres"

# -------- 1) Resolve required columns (best-effort auto-detect) --------
def pick(col_patterns):
    for p in col_patterns:
        for c in df.columns:
            if p in c:
                return c
    raise KeyError(f"Could not find any of columns with patterns: {col_patterns}")

mileage_col = pick('Mileage')
price_col   = pick('Price')
reg_col     = pick('Year')
engine_col  = pick('Engine')

# -------- 2) Convert messy text fields to numeric --------
# Registration year like "2025(25)" -> 2025
df[reg_col] = (
    df[reg_col].astype(str).str.extract(r'(\d{4})')[0].astype(float)
)

# Engine like "1.5L" -> 1.5
df[engine_col] = (
    df[engine_col].astype(str).str.extract(r'([\d.]+)')[0].astype(float)
)

# Price & mileage to numeric (if not already)
df[price_col]   = pd.to_numeric(df[price_col], errors='coerce')
df[mileage_col] = pd.to_numeric(df[mileage_col], errors='coerce')

# -------- 3) Mileage bands + YOUR average prices --------
bins   = [0, 20000, 40000, 60000, np.inf]
labels = ['0-20000', '20000-40000', '40000-60000', '60000+']
df['mileage_band'] = pd.cut(df[mileage_col], bins=bins, labels=labels, right=False)

avg_prices_by_band = (
    df.groupby('mileage_band', observed=True)[price_col]
      .mean()
      .round(2)
)
print("Average price by mileage band (computed from your data):")
print(avg_prices_by_band)
avg_prices_dict = avg_prices_by_band.to_dict()

# -------- 4) OLS regression (Price ~ Mileage + RegYear + EngineLitres) --------
# Remove rows with missing predictors/target
reg_data = df[[price_col, mileage_col, reg_col, engine_col]].dropna()
X = reg_data[[mileage_col, reg_col, engine_col]]
X = sm.add_constant(X)  # intercept
y = reg_data[price_col]

model = sm.OLS(y, X).fit()
print("\nOLS Summary (key stats):")
print(model.summary().tables[0])  # concise header table
print(model.summary().tables[1])  # coefficients table

# Predicted prices for all rows where features exist
df['predicted_price'] = np.nan
df.loc[reg_data.index, 'predicted_price'] = model.predict(X)

# -------- 5) One-sample t-tests: each band vs overall mean price --------
overall_mean = df[price_col].mean()
rows = []
for band in labels:
    sample = df.loc[df['mileage_band'] == band, price_col].dropna()
    if len(sample) >= 2:
        t_stat, p_val = ttest_1samp(sample, popmean=overall_mean, nan_policy='omit')
    else:
        t_stat, p_val = np.nan, np.nan
    rows.append({
        'mileage_band': band,
        'n': int(len(sample)),
        'band_mean': avg_prices_dict.get(band, np.nan),
        'overall_mean': round(overall_mean, 2) if pd.notna(overall_mean) else np.nan,
        't_stat': t_stat,
        'p_value': p_val
    })

t_results = pd.DataFrame(rows)
print("\nOne-sample t-tests: band mean vs overall mean price")
print(t_results)

# -------- 6) (Optional) If you want to compare against model's mean prediction instead --------
# model_mean = df['predicted_price'].mean()
# ... then replace 'overall_mean' with 'model_mean' in the t-test block above.

TypeError: pick() takes 1 positional argument but 4 were given